# Semantic Labs Metadata Export - Finance Report
This notebook extracts semantic metadata from the **Finance Report** semantic model in workspace **DDiB-FSI** and writes a JSON file to Lakehouse Files.

In [ ]:
%pip install semantic-link-labs

In [ ]:
import json
from datetime import datetime, timezone
from sempy_labs.tom import connect_semantic_model

In [ ]:
WORKSPACE = "DDiB-FSI"
DATASET = "Finance Report"
OUT_PATH = "/lakehouse/default/Files/finance_report_semantic_metadata.json"

In [ ]:
def _safe_str(value):
    if value is None:
        return ""
    try:
        return str(value)
    except Exception:
        return ""


def _iter_collection(obj):
    if obj is None:
        return []
    try:
        return list(obj)
    except Exception:
        return []

In [ ]:
tables_out = []
measures_out = []
relationships_out = []
roles_out = []

with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE) as tom:
    model = tom.model

    for table in _iter_collection(getattr(model, "Tables", None)):
        t_name = _safe_str(getattr(table, "Name", None))
        t_desc = _safe_str(getattr(table, "Description", None))
        t_hidden = bool(getattr(table, "IsHidden", False))

        cols = []
        for col in _iter_collection(getattr(table, "Columns", None)):
            cols.append({
                "name": _safe_str(getattr(col, "Name", None)),
                "description": _safe_str(getattr(col, "Description", None)),
                "dataType": _safe_str(getattr(col, "DataType", None)),
                "isHidden": bool(getattr(col, "IsHidden", False)),
                "formatString": _safe_str(getattr(col, "FormatString", None)),
            })

        hierarchies = []
        for h in _iter_collection(getattr(table, "Hierarchies", None)):
            levels = []
            for lvl in _iter_collection(getattr(h, "Levels", None)):
                levels.append({
                    "name": _safe_str(getattr(lvl, "Name", None)),
                    "column": _safe_str(getattr(getattr(lvl, "Column", None), "Name", None)),
                })
            hierarchies.append({
                "name": _safe_str(getattr(h, "Name", None)),
                "description": _safe_str(getattr(h, "Description", None)),
                "levels": levels,
            })

        for m in _iter_collection(getattr(table, "Measures", None)):
            measures_out.append({
                "table": t_name,
                "name": _safe_str(getattr(m, "Name", None)),
                "description": _safe_str(getattr(m, "Description", None)),
                "expression": _safe_str(getattr(m, "Expression", None)),
                "formatString": _safe_str(getattr(m, "FormatString", None)),
                "displayFolder": _safe_str(getattr(m, "DisplayFolder", None)),
                "isHidden": bool(getattr(m, "IsHidden", False)),
            })

        tables_out.append({
            "name": t_name,
            "description": t_desc,
            "isHidden": t_hidden,
            "columns": cols,
            "hierarchies": hierarchies,
        })

    for r in _iter_collection(getattr(model, "Relationships", None)):
        from_col = getattr(r, "FromColumn", None)
        to_col = getattr(r, "ToColumn", None)
        relationships_out.append({
            "name": _safe_str(getattr(r, "Name", None)),
            "description": _safe_str(getattr(r, "Description", None)),
            "fromTable": _safe_str(getattr(getattr(from_col, "Table", None), "Name", None)),
            "fromColumn": _safe_str(getattr(from_col, "Name", None)),
            "toTable": _safe_str(getattr(getattr(to_col, "Table", None), "Name", None)),
            "toColumn": _safe_str(getattr(to_col, "Name", None)),
            "crossFilteringBehavior": _safe_str(getattr(r, "CrossFilteringBehavior", None)),
            "isActive": bool(getattr(r, "IsActive", True)),
        })

    for role in _iter_collection(getattr(model, "Roles", None)):
        table_perms = []
        for p in _iter_collection(getattr(role, "TablePermissions", None)):
            table_perms.append({
                "table": _safe_str(getattr(getattr(p, "Table", None), "Name", None)),
                "filterExpression": _safe_str(getattr(p, "FilterExpression", None)),
            })
        roles_out.append({
            "name": _safe_str(getattr(role, "Name", None)),
            "description": _safe_str(getattr(role, "Description", None)),
            "modelPermission": _safe_str(getattr(role, "ModelPermission", None)),
            "tablePermissions": table_perms,
        })

In [ ]:
payload = {
    "source": "semantic-link-labs",
    "generatedAtUtc": datetime.now(timezone.utc).isoformat(),
    "workspace": WORKSPACE,
    "dataset": DATASET,
    "tables": tables_out,
    "measures": measures_out,
    "relationships": relationships_out,
    "roles": roles_out,
}

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"Wrote: {OUT_PATH}")
print(f"Tables: {len(payload['tables'])}")
print(f"Measures: {len(payload['measures'])}")
print(f"Relationships: {len(payload['relationships'])}")
print(f"Roles: {len(payload['roles'])}")

## Next Step
Download `/lakehouse/default/Files/finance_report_semantic_metadata.json` and run in this repo:

```powershell
.\import_semantic_labs_metadata_to_purview.ps1 -MetadataJsonPath ".\docs\finance_report_semantic_metadata.json"
```